## Mapping and standardization of literals to ICD-10 alphanumeric codes

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
preprocessed_icd = pd.read_csv('data/preprocessed/icd_preprocessed_descriptions.csv').dropna()
df_pcross = pd.read_csv('data/utils/preprocessed_cross.csv')

In [3]:
preprocessed_icd.head()

,processed,Code,D_P
0,colera,A00,D
1,"colera debido a vibrio cholerae 01, biotipo ch...",A000,D
2,"colera debido a vibrio cholerae 01, biotipo el...",A001,D
3,"colera, no especificado",A009,D
4,fiebres tifoidea y paratifoidea,A01,D


In [4]:
df_pcross.head()

,Code,D_P,processed_Description,processed_Literal
0,A020,D,enteritis debida a salmonella,gastroenteritis aguda por salmonella enterica
1,A045,D,enteritis debida a campylobacter,campylobacter jejuni
2,A080,D,enteritis debida a rotavirus,gastroenteritis aguda rotavirus
3,A080,D,enteritis debida a rotavirus,rotavirus
4,A09,D,"gastroenteritis y colitis infecciosas, no espe...",a09


In [5]:
# Setup KB Matrix (from preliminary_model.ipynb)
kb_vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=10000)
kb_matrix = kb_vectorizer.fit_transform(preprocessed_icd['processed'])

In [6]:
# Filter cross_referenced_data.csv
valid_icd10 = df_pcross[df_pcross['Code'].str.contains('^[a-zA-Z]', na=False)].copy()
to_rescue = df_pcross[~df_pcross['Code'].str.contains('^[a-zA-Z]', na=False)].copy()
to_rescue.head()

,Code,D_P,processed_Description,processed_Literal
7932,009U3ZX,P,"drenaje en canal espinal, diagnostico, abordaj...",puncio lumbar
7933,009U3ZX,P,"drenaje en canal espinal, diagnostico, abordaj...",liquido cefaloraquideo
7934,009U3ZX,P,"drenaje en canal espinal, diagnostico, abordaj...",lcr
7935,009U3ZX,P,"drenaje en canal espinal, diagnostico, abordaj...",puncion lumbar
7936,009U3ZX,P,"drenaje en canal espinal, diagnostico, abordaj...",puncion lumbar


In [7]:
# Match numeric literals to alphanumeric codes
rescued_codes = []
for literal in to_rescue['processed_Literal']:
    query_vec = kb_vectorizer.transform([str(literal)])
    similarities = cosine_similarity(query_vec, kb_matrix).flatten()
    best_idx = similarities.argmax()
    rescued_codes.append(preprocessed_icd.iloc[best_idx]['Code'])

to_rescue['Code'] = rescued_codes
len(to_rescue)

1579

In [8]:
to_rescue.head()

,Code,D_P,processed_Description,processed_Literal
7932,M4306,P,"drenaje en canal espinal, diagnostico, abordaj...",puncio lumbar
7933,Y651,P,"drenaje en canal espinal, diagnostico, abordaj...",liquido cefaloraquideo
7934,A00,P,"drenaje en canal espinal, diagnostico, abordaj...",lcr
7935,G971,P,"drenaje en canal espinal, diagnostico, abordaj...",puncion lumbar
7936,G971,P,"drenaje en canal espinal, diagnostico, abordaj...",puncion lumbar


In [9]:
# Combine
df_final_training = pd.concat([valid_icd10, to_rescue], ignore_index=True)

# Write to csv
df_final_training.to_csv('data/utils/icd10_training_set.csv', index=False)

In [10]:
len(df_final_training)

9943

To preserve all training data, 1579 literals (16%) from `cross_referenced_data.csv` were remapped to the ICD-10 standard using the preliminary model's similarity matrix due to old standard coding.